# Download CelebA – wspólna pula obrazów

Pobiera obrazy z **oficjalnych** splitów train/test CelebA (`flwrlabs/celeba`).

Zamiast uruchamiać pobieranie osobno dla każdego konceptu, skrypt kolekcjonuje dane w **jednym przebiegu**:
- obraz jest zapisywany, gdy jest potrzebny dla **co najmniej jednego** konceptu (brakuje pos lub neg),
- metadata zawierają **wszystkie 40 atrybutów** oryginalnego zbioru CelebA, nie tylko śledzone koncepty,
- skrypt zatrzymuje się, gdy **wszystkie** koncepty mają wystarczająco dużo obrazów (pos i neg — oddzielne liczniki),
- przy wznowieniu kompletne splity są pomijane; niekompletne są pobierane od nowa.

| split | źródło HF | cel na (koncept, label) |
|-------|-----------|-------------------------|
| train | `split="train"` | `7 × DS_SIZE / 2` |
| test  | `split="test"`  | `1 × DS_SIZE / 2` |

Obrazy: `data/images/`  |  Metadane: `data/metadata.csv`

In [1]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

In [2]:
DS_SIZE = 256

# Koncepty do zbierania: nazwa atrybutu HF -> kolumna w metadata
CONCEPTS = {
    'Eyeglasses':   'eyeglasses',
    'Bald':         'bald',
    'Chubby':       'chubby',
    'Wearing_Hat':  'wearing_hat',
    'Male':         'male',
}
_COL_TO_HF = {v: k for k, v in CONCEPTS.items()}

TRAIN_TARGET = 7 * DS_SIZE // 2   # 896 pos i 896 neg na każdy koncept
TEST_TARGET  = 1 * DS_SIZE // 2   # 128 pos i 128 neg na każdy koncept

ROOT = Path('..').resolve()
IMAGES_DIR    = ROOT / 'data' / 'images'
METADATA_PATH = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY = 500   # zapis metadanych co N zapisanych obrazów

print(f'Koncepty           : {list(CONCEPTS.values())}')
print(f'Train cel (pos/neg): {TRAIN_TARGET} na koncept')
print(f'Test  cel (pos/neg): {TEST_TARGET}  na koncept')
print(f'Folder obrazów     : {IMAGES_DIR}')
print(f'Metadane           : {METADATA_PATH}')

Koncepty           : ['eyeglasses', 'bald', 'chubby', 'wearing_hat', 'male']
Train cel (pos/neg): 896 na koncept
Test  cel (pos/neg): 128  na koncept
Folder obrazów     : E:\Do gita\WB2\data\images
Metadane           : E:\Do gita\WB2\data\metadata.csv


In [3]:
# Pełna lista 40 atrybutów CelebA zapisywanych w metadata
# klucz = nazwa w datasecie HF, wartość = nazwa kolumny CSV
ALL_CELEBA_ATTRS = {
    '5_o_Clock_Shadow':    '5_o_clock_shadow',
    'Arched_Eyebrows':     'arched_eyebrows',
    'Attractive':          'attractive',
    'Bags_Under_Eyes':     'bags_under_eyes',
    'Bald':                'bald',
    'Bangs':               'bangs',
    'Big_Lips':            'big_lips',
    'Big_Nose':            'big_nose',
    'Black_Hair':          'black_hair',
    'Blond_Hair':          'blond_hair',
    'Blurry':              'blurry',
    'Brown_Hair':          'brown_hair',
    'Bushy_Eyebrows':      'bushy_eyebrows',
    'Chubby':              'chubby',
    'Double_Chin':         'double_chin',
    'Eyeglasses':          'eyeglasses',
    'Goatee':              'goatee',
    'Gray_Hair':           'gray_hair',
    'Heavy_Makeup':        'heavy_makeup',
    'High_Cheekbones':     'high_cheekbones',
    'Male':                'male',
    'Mouth_Slightly_Open': 'mouth_slightly_open',
    'Mustache':            'mustache',
    'Narrow_Eyes':         'narrow_eyes',
    'No_Beard':            'no_beard',
    'Oval_Face':           'oval_face',
    'Pale_Skin':           'pale_skin',
    'Pointy_Nose':         'pointy_nose',
    'Receding_Hairline':   'receding_hairline',
    'Rosy_Cheeks':         'rosy_cheeks',
    'Sideburns':           'sideburns',
    'Smiling':             'smiling',
    'Straight_Hair':       'straight_hair',
    'Wavy_Hair':           'wavy_hair',
    'Wearing_Earrings':    'wearing_earrings',
    'Wearing_Hat':         'wearing_hat',
    'Wearing_Lipstick':    'wearing_lipstick',
    'Wearing_Necklace':    'wearing_necklace',
    'Wearing_Necktie':     'wearing_necktie',
    'Young':               'young',
}

print(f'Atrybuty w metadata: {len(ALL_CELEBA_ATTRS)}')

Atrybuty w metadata: 40


In [4]:
def make_counts():
    """Inicjalizuje zerowe liczniki pos/neg dla każdego konceptu i splitu."""
    return {
        col: {'train': {'pos': 0, 'neg': 0}, 'test': {'pos': 0, 'neg': 0}}
        for col in CONCEPTS.values()
    }

def counts_from_df(df):
    """Odtwarza liczniki z istniejącego DataFrame."""
    counts = make_counts()
    for split in ('train', 'test'):
        sub = df[df['split'] == split]
        for col in CONCEPTS.values():
            if col not in sub.columns:
                continue
            counts[col][split]['pos'] = int((sub[col] == 1).sum())
            counts[col][split]['neg'] = int((sub[col] == 0).sum())
    return counts

def _target(split_label):
    return TRAIN_TARGET if split_label == 'train' else TEST_TARGET

def is_useful(item, split_label, counts):
    """True jeśli obraz wnosi coś do co najmniej jednego konceptu w tym splicie."""
    tgt = _target(split_label)
    for hf_attr, col in CONCEPTS.items():
        key = 'pos' if bool(item[hf_attr]) else 'neg'
        if counts[col][split_label][key] < tgt:
            return True
    return False

def is_split_done(split_label, counts):
    """True gdy wszystkie koncepty mają wystarczająco dużo pos i neg w tym splicie."""
    tgt = _target(split_label)
    return all(
        counts[col][split_label]['pos'] >= tgt and counts[col][split_label]['neg'] >= tgt
        for col in CONCEPTS.values()
    )

def show_status(counts):
    """Wyświetla tabelę stanu zbierania danych per koncept."""
    rows = []
    for col in CONCEPTS.values():
        tp = counts[col]['train']['pos']
        tn = counts[col]['train']['neg']
        ep = counts[col]['test']['pos']
        en = counts[col]['test']['neg']
        train_ok = 'OK' if tp >= TRAIN_TARGET and tn >= TRAIN_TARGET else '--'
        test_ok  = 'OK' if ep >= TEST_TARGET  and en >= TEST_TARGET  else '--'
        rows.append({
            'concept':    col,
            'train_pos':  f'{tp}/{TRAIN_TARGET}',
            'train_neg':  f'{tn}/{TRAIN_TARGET}',
            'train':      train_ok,
            'test_pos':   f'{ep}/{TEST_TARGET}',
            'test_neg':   f'{en}/{TEST_TARGET}',
            'test':       test_ok,
        })
    print(pd.DataFrame(rows).to_string(index=False))

In [5]:
metadata_list = []
counts = make_counts()
next_idx = 0

if METADATA_PATH.exists():
    existing_df = pd.read_csv(METADATA_PATH)
    all_counts  = counts_from_df(existing_df)

    # Zachowaj tylko splity, które są już kompletne
    complete = [s for s in ('train', 'test') if is_split_done(s, all_counts)]
    incomplete = [s for s in ('train', 'test') if s not in complete]

    if complete:
        kept_df = existing_df[existing_df['split'].isin(complete)].copy()
        metadata_list = kept_df.to_dict('records')
        counts = counts_from_df(kept_df)
        print(f'Wczytano {len(kept_df)} obrazów z kompletnych splitów: {complete}')
    else:
        print('Brak kompletnych splitów.')

    # Usuń pliki obrazów z niekompletnych splitów
    if incomplete:
        to_remove = existing_df[existing_df['split'].isin(incomplete)]['filename'].tolist()
        removed = 0
        for fn in to_remove:
            fp = IMAGES_DIR / fn
            if fp.exists():
                fp.unlink()
                removed += 1
        if removed:
            print(f'Usunięto {removed} obrazów z niekompletnych splitów: {incomplete}')

    next_idx = len(metadata_list)
else:
    print('Brak istniejących metadanych — pobieranie od zera.')

print()
show_status(counts)

Brak istniejących metadanych — pobieranie od zera.

    concept train_pos train_neg train test_pos test_neg test
 eyeglasses     0/896     0/896    --    0/128    0/128   --
       bald     0/896     0/896    --    0/128    0/128   --
     chubby     0/896     0/896    --    0/128    0/128   --
wearing_hat     0/896     0/896    --    0/128    0/128   --
       male     0/896     0/896    --    0/128    0/128   --


In [6]:
def collect_split(hf_split, split_label, counts, metadata_list, start_idx):
    if is_split_done(split_label, counts):
        print(f'[{split_label}] Już kompletny — pomijam.')
        return start_idx

    tgt = _target(split_label)
    print(f'\n[{split_label}] Cel: {tgt} pos + {tgt} neg na każdy z {len(CONCEPTS)} konceptów')

    ds = load_dataset('flwrlabs/celeba', split=hf_split, streaming=True, token=HF_TOKEN)
    idx = start_idx
    saved = 0

    for item in tqdm(ds, desc=split_label):
        if is_split_done(split_label, counts):
            break

        if not is_useful(item, split_label, counts):
            continue

        # Zapisz obraz
        img = item['image'].convert('RGB').resize((224, 224))
        filename = f'img_{idx:06d}.jpg'
        img.save(IMAGES_DIR / filename, quality=90)

        # Wiersz metadanych ze WSZYSTKIMI atrybutami CelebA
        row = {'filename': filename, 'split': split_label}
        for hf_attr, col in ALL_CELEBA_ATTRS.items():
            row[col] = int(item[hf_attr]) if hf_attr in item else None
        metadata_list.append(row)

        # Aktualizuj liczniki wszystkich śledzonych konceptów
        for hf_attr, col in CONCEPTS.items():
            key = 'pos' if bool(item[hf_attr]) else 'neg'
            counts[col][split_label][key] += 1

        idx += 1
        saved += 1

        if saved % CHECKPOINT_EVERY == 0:
            pd.DataFrame(metadata_list).to_csv(METADATA_PATH, index=False)

    print(f'  -> zapisano {saved} nowych obrazów (łącznie w pliku: {idx})')
    show_status(counts)
    return idx

In [7]:
idx = next_idx
idx = collect_split('train', 'train', counts, metadata_list, idx)
idx = collect_split('test',  'test',  counts, metadata_list, idx)

# Zapis końcowy
df_final = pd.DataFrame(metadata_list)
col_order = ['filename', 'split'] + list(ALL_CELEBA_ATTRS.values())
col_order = [c for c in col_order if c in df_final.columns]
df_final = df_final[col_order]
df_final.to_csv(METADATA_PATH, index=False)
print(f'\nZapisano {len(df_final)} rekordów lacznie -> {METADATA_PATH}')


[train] Cel: 896 pos + 896 neg na każdy z 5 konceptów


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

train: 39623it [06:37, 99.64it/s] 


  -> zapisano 4642 nowych obrazów (łącznie w pliku: 4642)
    concept train_pos train_neg train test_pos test_neg test
 eyeglasses  1103/896  3539/896    OK    0/128    0/128   --
       bald   896/896  3746/896    OK    0/128    0/128   --
     chubby  1182/896  3460/896    OK    0/128    0/128   --
wearing_hat   896/896  3746/896    OK    0/128    0/128   --
       male  3110/896  1532/896    OK    0/128    0/128   --

[test] Cel: 128 pos + 128 neg na każdy z 5 konceptów


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

test: 8449it [01:26, 97.51it/s] 

  -> zapisano 715 nowych obrazów (łącznie w pliku: 5357)
    concept train_pos train_neg train test_pos test_neg test
 eyeglasses  1103/896  3539/896    OK  172/128  543/128   OK
       bald   896/896  3746/896    OK  128/128  587/128   OK
     chubby  1182/896  3460/896    OK  161/128  554/128   OK
wearing_hat   896/896  3746/896    OK  128/128  587/128   OK
       male  3110/896  1532/896    OK  458/128  257/128   OK

Zapisano 5357 rekordów lacznie -> E:\Do gita\WB2\data\metadata.csv


## Podsumowanie pobranych danych

In [8]:
df = pd.read_csv(METADATA_PATH)
print(f'Lacznie obrazow: {len(df)}')
print(f'Kolumny: {list(df.columns)}')
print(f'\nPodzial per split:')
print(df['split'].value_counts())

print(f'\nStatus konceptow:')
show_status(counts_from_df(df))

print(f'\n% pozytywnych przypadkow sledzonch konceptow per split:')
concept_cols = list(CONCEPTS.values())
summary = (
    df.groupby('split')[concept_cols]
    .mean()
    .mul(100)
    .round(1)
)
summary.columns.name = 'concept (% pozytywnych)'
display(summary)

Lacznie obrazow: 5357
Kolumny: ['filename', 'split', '5_o_clock_shadow', 'arched_eyebrows', 'attractive', 'bags_under_eyes', 'bald', 'bangs', 'big_lips', 'big_nose', 'black_hair', 'blond_hair', 'blurry', 'brown_hair', 'bushy_eyebrows', 'chubby', 'double_chin', 'eyeglasses', 'goatee', 'gray_hair', 'heavy_makeup', 'high_cheekbones', 'male', 'mouth_slightly_open', 'mustache', 'narrow_eyes', 'no_beard', 'oval_face', 'pale_skin', 'pointy_nose', 'receding_hairline', 'rosy_cheeks', 'sideburns', 'smiling', 'straight_hair', 'wavy_hair', 'wearing_earrings', 'wearing_hat', 'wearing_lipstick', 'wearing_necklace', 'wearing_necktie', 'young']

Podzial per split:
split
train    4642
test      715
Name: count, dtype: int64

Status konceptow:
    concept train_pos train_neg train test_pos test_neg test
 eyeglasses  1103/896  3539/896    OK  172/128  543/128   OK
       bald   896/896  3746/896    OK  128/128  587/128   OK
     chubby  1182/896  3460/896    OK  161/128  554/128   OK
wearing_hat   896/89

concept (% pozytywnych),eyeglasses,bald,chubby,wearing_hat,male
split,,,,,
test,24.1,17.9,22.5,17.9,64.1
train,23.8,19.3,25.5,19.3,67.0
